In [ ]:
from utils.analysis.valuation import BuySellSignalsAnalyzer, SignalsReporter
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE, TRADING_SIGNALS_CONFIG

In [ ]:
# 📊 CONFIGURACIÓN
# Las fechas vienen de config.py (TRADING_SIGNALS_CONFIG)
# Personaliza aquí solo si necesitas valores diferentes

# Tickers a analizar
TICKERS = ["INCY"]  # Agrega más: ["AAPL", "MSFT", "GOOGL", ...]

# Lookback para análisis de precios
LOOKBACK_YEARS = 5  # Años de histórico

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
CUSTOM_START = ""  # Ej: "2022-01-01"
CUSTOM_END = ""    # Ej: "2024-12-31"

print("📋 Configuración:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Lookback: {LOOKBACK_YEARS} años")
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    print(f"   Fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    print(f"   Fechas desde config.py: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE}")

In [ ]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager
data_manager = DataManager()

# Crear analyzer con configuración
# Si USE_CUSTOM_DATES es True, usar fechas personalizadas
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        start_date=CUSTOM_START,
        end_date=CUSTOM_END,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con configuración de config.py")

reporter = SignalsReporter()
print("✅ Reporter inicializado")

In [ ]:
# 🔍 ANÁLISIS DE SEÑALES
signals = []

print(f"\n🔍 Analizando {len(TICKERS)} ticker(s)...\n")

for ticker in TICKERS:
    try:
        # El analyzer ya tiene las fechas configuradas
        signal = analyzer.analyze_stock(ticker)
        signals.append(signal)
        
        # Mostrar resumen rápido
        emoji = "🟢" if signal.signal == "COMPRA" else "🔴" if signal.signal == "VENTA" else "🟡"
        print(f"{emoji} {ticker}: {signal.signal} "
              f"(Conf: {signal.confidence:.1%}, Pot: {signal.upside_potential:+.1f}%)")
        
    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)[:60]}")

print(f"\n📊 Total analizado: {len(signals)}/{len(TICKERS)} tickers")

Período: 2020-01-01 → 2025-12-22


[*********************100%***********************]  1 of 1 completed

✅ INCY analizado


In [ ]:
# 📊 RESUMEN EN TABLA
if signals:
    summary_df = reporter.to_dataframe(signals)
    display(summary_df)
    
    print()
    reporter.print_summary(signals)
else:
    print("❌ No hay señales para mostrar")

,Ticker,Señal,Confianza,Valoración,Fundamental,Precio Actual,Precio Objetivo,Potencial
0,INCY,COMPRA,64.1%,59.2,93.2,$101,$100,-0.6%



RESUMEN DE SEÑALES

🟢 COMPRAS: 1
   INCY: 64.1% confianza, -0.6% potencial

🔴 VENTAS: 0

🟡 MANTENER: 0


In [ ]:
# 🏆 TOP OPORTUNIDADES DE COMPRA
if signals:
    reporter.print_top_opportunities(signals, top_n=5)
else:
    print("❌ No hay señales para analizar")


TOP 3 OPORTUNIDADES DE COMPRA

1. INCY
   Confianza: 64.1%
   Potencial: -0.6%
   Precio: $101 → $100
   Valoración: 59.2 | Fundamental: 93.2
   Razones: 💵 Valoración razonable - Precio cerca del valor justo, ⭐ Empresa de calidad excepcional


In [ ]:
# 🟢 DETALLE DE SEÑALES DE COMPRA
compras = [s for s in signals if s.signal == "COMPRA"]

if compras:
    # Ordenar por confianza
    top_compras = sorted(compras, key=lambda x: x.confidence, reverse=True)
    
    print(f"📈 Encontradas {len(compras)} señales de COMPRA\n")
    print("=" * 65)
    
    for signal in top_compras:
        reporter.print_signal(signal)
        print()
else:
    print("ℹ️  No hay señales de COMPRA en este momento")

SEÑAL DE INVERSIÓN: INCY

🟢 COMPRA (Confianza: 64.1%)
📊 SCORES:
   Valoración:    🟠 [███████████░░░░░░░░░] 59.2
   Fundamental:   🟢 [██████████████████░░] 93.2
💰 PRECIOS:
   Actual:        $101
   Objetivo:      $100
   Potencial:     -0.6%

💡 RAZONES:
   💵 Valoración razonable - Precio cerca del valor justo
   ⭐ Empresa de calidad excepcional
   📈 Excelente rentabilidad
   🛡️ Balance sólido - Baja deuda
   🚀 Alto crecimiento sostenido


In [ ]:
# 🏭 ANÁLISIS POR SECTOR
# Analiza múltiples tickers de un sector específico

TECH_SECTOR = ["AAPL", "MSFT", "GOOGL", "META", "NVDA", "AMD", "INTC", "CRM"]

print("🔍 Analizando sector tecnológico...\n")
tech_signals = []

for ticker in TECH_SECTOR:
    try:
        signal = analyzer.analyze_stock(ticker)
        tech_signals.append(signal)
    except Exception as e:
        print(f"⚠️  {ticker}: {str(e)[:40]}")

if tech_signals:
    # Mostrar solo las mejores oportunidades
    compras_tech = [s for s in tech_signals if s.signal == "COMPRA"]
    
    if compras_tech:
        print(f"\n✅ {len(compras_tech)} oportunidades de compra en Tech:\n")
        
        tech_df = reporter.to_dataframe(compras_tech)
        display(tech_df.sort_values('Confianza', ascending=False))
    else:
        print("\nℹ️  No hay señales de compra en el sector tech actualmente")

In [ ]:
# 📊 ANÁLISIS DE SENSIBILIDAD TEMPORAL
# Compara señales con diferentes períodos de lookback

TICKER_TEST = "AAPL"
PERIODS = [(1, "1 año"), (3, "3 años"), (5, "5 años")]

print(f"🔬 Análisis de sensibilidad temporal para {TICKER_TEST}:\n")
print(f"{'Período':<10} {'Señal':<10} {'Confianza':<12} {'Valoración':<12} {'Fundamental':<12} {'Potencial'}")
print("-" * 75)

for years, label in PERIODS:
    try:
        test_analyzer = BuySellSignalsAnalyzer(
            data_manager=data_manager,
            lookback_years=years
        )
        signal = test_analyzer.analyze_stock(TICKER_TEST)
        
        print(f"{label:<10} {signal.signal:<10} {signal.confidence:>5.1%}      "
              f"{signal.valuation_score:>6.1f}      "
              f"{signal.fundamental_score:>6.1f}      "
              f"{signal.upside_potential:>+6.1f}%")
        
    except Exception as e:
        print(f"{label:<10} Error: {str(e)[:45]}")

print("\n💡 Observa cómo cambia la señal según el período analizado")